# SDC380 — 4.6 Project: Stroke Prediction Using Python
**Student:** Brendan Tice  
**Student ID:** BRETIC0307  
**Date:** November 1, 2026

Logistic regression classification of Stroke vs No Stroke. Data source: provided Stroke Prediction Dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score, mean_absolute_error, mean_squared_error, r2_score, classification_report, confusion_matrix)

In [ ]:
df = pd.read_excel("Stroke_Prediction_Dataset(3).xlsx")
print("Shape:", df.shape)
print("Missing values per column:\n", df.isna().sum().to_string())
print("Diagnosis distribution:\n", df["Diagnosis"].value_counts())
display(df.head())

In [ ]:
y = df["Diagnosis"].astype(str).str.strip().str.lower().eq("stroke").astype(int)
X = df.drop(columns=["Diagnosis", "Patient ID", "Patient Name"])
X = pd.get_dummies(X, drop_first=False, dtype=int)
X = X.apply(pd.to_numeric, errors="coerce")
X = X.fillna(X.median()).fillna(0)
print("Encoded feature count:", X.shape[1])
print("Remaining nulls:", X.isna().sum().sum())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print("Training:", X_train.shape, "Testing:", X_test.shape)

In [ ]:
model = Pipeline([("scale", StandardScaler()), ("logreg", LogisticRegression(max_iter=3000, class_weight="balanced", random_state=42))])
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall:", recall_score(y_test, y_pred, zero_division=0))
print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("R²:", r2_score(y_test, y_pred))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=0))
# R² is included for the rubric but is not a recommended classification metric.

In [ ]:
coefs = pd.Series(model.named_steps["logreg"].coef_[0], index=X.columns).sort_values(key=abs, ascending=False)
print("Feature ranking (standardized coefficients):")
print(coefs.head(20).to_string())
coefs.head(12).sort_values().plot.barh(figsize=(9,5), title="Top 12 Logistic Regression Coefficients")
plt.xlabel("Standardized coefficient")
plt.tight_layout()
plt.show()

In [ ]:
baseline = X.median(numeric_only=True).reindex(X.columns).fillna(0).to_frame().T
bmi_values = [20, 25, 30, 35, 40]
bmi_probs = []
for bmi_value in bmi_values:
    row = baseline.copy()
    row["Body Mass Index (BMI)"] = bmi_value
    bmi_probs.append(model.predict_proba(row)[0, 1])
print(pd.DataFrame({"BMI": bmi_values, "Predicted stroke probability": bmi_probs}).to_string(index=False))
plt.plot(bmi_values, bmi_probs, marker="o")
plt.xlabel("BMI")
plt.ylabel("Predicted probability of stroke")
plt.ylim(0,1)
plt.title("BMI What-if Analysis")
plt.show()

**Interpretation:** Coefficients indicate associations in the fitted model, not causal effects. The BMI what-if holds other encoded inputs at their median values, which may not represent a real patient. Class imbalance makes precision and recall especially important. The dataset is for coursework, not clinical decision-making.